## Expanded out-of-sample accuracy diagnostics

In [2]:
import os
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import KFold
from tensorflow.keras.optimizers import Adam

import itertools
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# --------------------------------------------------
# Set random seed for reproducibility
# --------------------------------------------------
SEED = 125

# Python
random.seed(SEED)

# NumPy
np.random.seed(SEED)

# TensorFlow/Keras
tf.keras.utils.set_random_seed(SEED)

# Enable deterministic TensorFlow operations
tf.config.experimental.enable_op_determinism()

print(f"Random seed set to {SEED}")

Random seed set to 125


## Model-Building Function
Create a function that builds a Keras model based on input parameters. This function allows flexibility in specifying the number of layers, neurons, activation functions, regularization, etc.

In [4]:
def build_model(input_dim,
                nr_neurons,
                reg_param=0.00,
                activation='relu',
                use_batch_norm=True,
                dropout_rate=0.0,
                learning_rate=0.005):
    """
    Builds a customizable neural network model for regression tasks.

    Parameters:
    - input_dim (int): Number of input features.
    - nr_neurons (list): List defining the number of neurons in each hidden layer.
                            Example: [64, 32] for 2 layers with 64 and 32 neurons.
    - reg_param (float): L2 regularization parameter.
    - activation (str): Activation function for hidden layers.
    - use_batch_norm (bool): Whether to use Batch Normalization.
    - dropout_rate (float): Dropout rate for regularization.

    Returns:
    - model (keras.Model): Compiled Keras model.
    """
    model = models.Sequential()

    # Input layer (first hidden layer)
    model.add(layers.Dense(nr_neurons[0],
                           kernel_regularizer=regularizers.l2(reg_param),
                           activation=activation,
                           input_shape=(input_dim,)))

    if use_batch_norm:
        model.add(layers.BatchNormalization())
    if dropout_rate > 0.0:
        model.add(layers.Dropout(dropout_rate))

    # Add remaining hidden layers
    for neurons in nr_neurons[1:]:
        model.add(layers.Dense(neurons,
                               kernel_regularizer=regularizers.l2(reg_param),
                               activation=activation))
        if use_batch_norm:
            model.add(layers.BatchNormalization())
        if dropout_rate > 0.0:
            model.add(layers.Dropout(dropout_rate))

    # Output layer
    model.add(layers.Dense(1))  # Single output for regression

    # Compile the model
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='mse',
                  metrics=['mae', 'mse'])

    return model

In [5]:
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True)

## Read the data - Gamma

In [63]:
# Load your dataset
df = pd.read_csv("CAT_price_gamma.csv")

print(df.head())

      c         r  kappa  theta  sigma     lambda             D  N         T  \
0  0.05  0.004795    0.2   0.03   0.02  30.210800  1.264946e+10  8  1.127421   
1  0.05  0.054865    0.2   0.03   0.02  39.169999  1.299456e+10  3  1.846108   
2  0.05  0.068230    0.2   0.03   0.02  34.941122  8.628679e+09  8  1.246646   
3  0.05  0.000815    0.2   0.03   0.02  35.785227  1.164647e+10  3  1.729329   
4  0.05  0.055641    0.2   0.03   0.02  32.626761  9.544387e+09  6  0.537173   

         Price  
0  1390.038266  
1   794.831852  
2  1148.438033  
3   936.035903  
4  1266.170898  


In [65]:
# Define your features (X) and labels (y)
# Replace 'feature_columns' and 'target_column' with actual column names
X = df[['r','lambda', 'D', 'N', 'T']]
y = df['Price']/1000
print(X.head())
print(y.head())

          r     lambda             D  N         T
0  0.004795  30.210800  1.264946e+10  8  1.127421
1  0.054865  39.169999  1.299456e+10  3  1.846108
2  0.068230  34.941122  8.628679e+09  8  1.246646
3  0.000815  35.785227  1.164647e+10  3  1.729329
4  0.055641  32.626761  9.544387e+09  6  0.537173
0    1.390038
1    0.794832
2    1.148438
3    0.936036
4    1.266171
Name: Price, dtype: float64


## Training on Entire Dataset

In [68]:
# Split data into training and testing sets
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=125, shuffle=True
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print("Min y_train:", np.min(y_train))
print("Max y_train:", np.max(y_train))

Min y_train: 0.0003526900584948981
Max y_train: 1.5997562713872224


### Test configuration 1

In [71]:
# Define different configurations
test_layers = [128, 64, 32]

test_activation = 'relu'
test_l2 = 1e-4
test_dropout = 0.1
test_batch_norm = True
test_learning = 1e-5

In [73]:
import time

EPOCHS = 1000
BATCH_SIZE = 512

# Build the model
test_model_gamma = build_model(input_dim=X_train.shape[1],
                         nr_neurons=test_layers,
                         reg_param=test_l2,
                         activation=test_activation,
                         use_batch_norm=test_batch_norm,
                         dropout_rate=test_dropout,
                         learning_rate=test_learning)
# Measure training time
start_time = time.perf_counter()

# Train the model on the entire dataset
history_gamma = test_model_gamma.fit(X_train, y_train,
                         epochs=EPOCHS,
                         batch_size=BATCH_SIZE,
                         validation_split=0.2,
                         shuffle=True,              
                         callbacks=[early_stop],
                         verbose=1)

training_time = time.perf_counter() - start_time

print(f"Training time: {training_time:.2f} seconds")

Epoch 1/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 2.9752 - mae: 1.3997 - mse: 2.9613 - val_loss: 1.2973 - val_mae: 1.0275 - val_mse: 1.2833
Epoch 2/1000
  1/750 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - loss: 1.8833 - mae: 1.1380 - mse: 1.8694

2026-07-04 11:26:11.316312: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-07-04 11:26:11.316800: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 1.7082 - mae: 1.0944 - mse: 1.6943 - val_loss: 1.0573 - val_mae: 0.9623 - val_mse: 1.0434
Epoch 3/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 1.4190 - mae: 1.0095 - mse: 1.4050 - val_loss: 0.9133 - val_mae: 0.9066 - val_mse: 0.8993
Epoch 4/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 1.2280 - mae: 0.9456 - mse: 1.2141 - val_loss: 0.8015 - val_mae: 0.8564 - val_mse: 0.7875
Epoch 5/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.0768 - mae: 0.8896 - mse: 1.0629 - val_loss: 0.7076 - val_mae: 0.8081 - val_mse: 0.6936
Epoch 6/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.9558 - mae: 0.8401 - mse: 0.9419 - val_loss: 0.6249 - val_mae: 0.7614 - val_mse: 0.6110
Epoch 7/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.8510 - mae: 0.7921 - mse: 0.8371 - val_loss: 0.5453 - val_mae: 0.7112 - val_mse: 0.5314
Epoch 8/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.7584 - mae: 0.7452 - mse: 0.7445 - va

In [74]:
# Predictions
y_pred = test_model_gamma.predict(X_test, verbose=0).flatten()

# Errors
error_gamma = y_pred - y_test
abs_error_gamma = np.abs(error_gamma)

# Statistics
results_gamma = {
    "Severity": "Gamma",
    "Observations": len(y_test),
    "Training Time (s)": training_time,
    "Bias": np.mean(error_gamma),
    "MAE": np.mean(abs_error_gamma),
    "MSE": np.mean(error_gamma**2),
    "RMSE": np.sqrt(np.mean(error_gamma**2)),
    "95% AE": np.quantile(abs_error_gamma, 0.95),
    "99% AE": np.quantile(abs_error_gamma, 0.99),
    "Max AE": np.max(abs_error_gamma),
    "R2": r2_score(y_test, y_pred)
}

results_gamma_df = pd.DataFrame([results_gamma])

print(results_gamma_df)

  Severity  Observations  Training Time (s)      Bias       MAE       MSE  \
0    Gamma        120000         653.691454  0.001151  0.003305  0.000021   

       RMSE    95% AE    99% AE    Max AE        R2  
0  0.004578  0.009719  0.014878  0.033084  0.999838  


### Test configuration 2

In [16]:
# Define different configurations
test_layers = [512, 256, 128, 64]

test_activation = 'relu'
test_l2 = 1e-4
test_dropout = 0.1
test_batch_norm = True
test_learning = 1e-5

In [17]:
import time

EPOCHS = 1000
BATCH_SIZE = 512

# Build the model
test_model_gamma = build_model(input_dim=X_train.shape[1],
                         nr_neurons=test_layers,
                         reg_param=test_l2,
                         activation=test_activation,
                         use_batch_norm=test_batch_norm,
                         dropout_rate=test_dropout,
                         learning_rate=test_learning)
# Measure training time
start_time = time.perf_counter()

# Train the model on the entire dataset
history_gamma = test_model_gamma.fit(X_train, y_train,
                         epochs=EPOCHS,
                         batch_size=BATCH_SIZE,
                         validation_split=0.2,
                         shuffle=True,             
                         callbacks=[early_stop],
                         verbose=1)

training_time = time.perf_counter() - start_time

print(f"Training time: {training_time:.2f} seconds")

Epoch 1/1000
743/750 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.0889 - mae: 1.1825 - mse: 2.0282

2026-07-02 10:28:27.026670: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-07-02 10:28:27.026906: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

750/750 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 2.0857 - mae: 1.1817 - mse: 2.0249 - val_loss: 0.9804 - val_mae: 0.9290 - val_mse: 0.9197
Epoch 2/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 1.3860 - mae: 1.0043 - mse: 1.3252 - val_loss: 0.8854 - val_mae: 0.8905 - val_mse: 0.8247
Epoch 3/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 1.1731 - mae: 0.9353 - mse: 1.1124 - val_loss: 0.7816 - val_mae: 0.8365 - val_mse: 0.7209
Epoch 4/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 1.0245 - mae: 0.8734 - mse: 0.9639 - val_loss: 0.6866 - val_mae: 0.7811 - val_mse: 0.6259
Epoch 5/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.9014 - mae: 0.8136 - mse: 0.8407 - val_loss: 0.5948 - val_mae: 0.7220 - val_mse: 0.5342
Epoch 6/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.7917 - mae: 0.7544 - mse: 0.7311 - val_loss: 0.4976 - val_mae: 0.6528 - val_mse: 0.4371
Epoch 7/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.6927 - mae: 0.6942 - mse: 0.6322 - va

In [18]:
# Predictions
y_pred = test_model_gamma.predict(X_test, verbose=0).flatten()

# Errors
error_gamma = y_pred - y_test
abs_error_gamma = np.abs(error_gamma)

# Statistics
results_gamma = {
    "Severity": "Gamma",
    "Observations": len(y_test),
    "Training Time (s)": training_time,
    "Bias": np.mean(error_gamma),
    "MAE": np.mean(abs_error_gamma),
    "MSE": np.mean(error_gamma**2),
    "RMSE": np.sqrt(np.mean(error_gamma**2)),
    "95% AE": np.quantile(abs_error_gamma, 0.95),
    "99% AE": np.quantile(abs_error_gamma, 0.99),
    "Max AE": np.max(abs_error_gamma),
    "R2": r2_score(y_test, y_pred)
}

results_gamma_df = pd.DataFrame([results_gamma])

print(results_gamma_df)

  Severity  Observations  Training Time (s)      Bias       MAE       MSE  \
0    Gamma        120000        2286.540794  0.000219  0.003172  0.000019   

       RMSE    95% AE    99% AE    Max AE        R2  
0  0.004374  0.009397  0.014554  0.026808  0.999852  


## Read the data - Lognormal

In [20]:
# Load your dataset
df = pd.read_csv("CAT_price_log.csv")

print(df.head())
print(df.shape)

      c         r  kappa  theta  sigma     lambda             D   N         T  \
0  0.05  0.047464    0.2   0.03   0.02  37.595368  1.067337e+10  10  1.971323   
1  0.05  0.005287    0.2   0.03   0.02  30.240079  1.244350e+10  10  1.947998   
2  0.05  0.037334    0.2   0.03   0.02  33.820770  9.731976e+09  10  1.557770   
3  0.05  0.026329    0.2   0.03   0.02  37.278229  1.171317e+10   8  0.815643   
4  0.05  0.046435    0.2   0.03   0.02  31.427952  1.049263e+10  12  0.832221   

         Price  
0   680.036064  
1  1384.209831  
2  1179.769620  
3  1373.370457  
4  1548.602842  
(600000, 10)


In [21]:
# Define your features (X) and labels (y)
# Replace 'feature_columns' and 'target_column' with actual column names
X = df[['r','lambda', 'D', 'N', 'T']]
y = df['Price'] / 1000

print(X.head())
print(y.head())

          r     lambda             D   N         T
0  0.047464  37.595368  1.067337e+10  10  1.971323
1  0.005287  30.240079  1.244350e+10  10  1.947998
2  0.037334  33.820770  9.731976e+09  10  1.557770
3  0.026329  37.278229  1.171317e+10   8  0.815643
4  0.046435  31.427952  1.049263e+10  12  0.832221
0    0.680036
1    1.384210
2    1.179770
3    1.373370
4    1.548603
Name: Price, dtype: float64


## Training on Entire Dataset

In [23]:
# Split data into training and testing sets
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=125, shuffle=True)

# Normalize the features using StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print("Min y_train:", np.min(y_train))
print("Max y_train:", np.max(y_train))

Min y_train: 0.0016483922476573647
Max y_train: 1.5996804426516338


### Test configuration 1

In [25]:
# Define different configurations
test_layers = [128, 64, 32]

test_activation = 'relu'
test_l2 = 1e-4
test_dropout = 0.1
test_batch_norm = True
test_learning = 1e-5

In [26]:
import time

EPOCHS = 1000
BATCH_SIZE = 512

# Build the model
test_model_log = build_model(input_dim=X_train.shape[1],
                         nr_neurons=test_layers,
                         reg_param=test_l2,
                         activation=test_activation,
                         use_batch_norm=test_batch_norm,
                         dropout_rate=test_dropout,
                         learning_rate=test_learning)
# Measure training time
start_time = time.perf_counter()

# Train the model on the entire dataset
history_log = test_model_log.fit(X_train, y_train,
                         epochs=EPOCHS,
                         batch_size=BATCH_SIZE,
                         validation_split=0.2,
                         shuffle=True,          
                         callbacks=[early_stop],
                         verbose=1)

training_time = time.perf_counter() - start_time

print(f"Training time: {training_time:.2f} seconds")

Epoch 1/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 2.9435 - mae: 1.3783 - mse: 2.9297 - val_loss: 1.2766 - val_mae: 1.0040 - val_mse: 1.2628
Epoch 2/1000
  1/750 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - loss: 1.7321 - mae: 1.0775 - mse: 1.7183

2026-07-02 11:06:32.170719: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-07-02 11:06:32.170965: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.7495 - mae: 1.0983 - mse: 1.7357 - val_loss: 0.9878 - val_mae: 0.9208 - val_mse: 0.9740
Epoch 3/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.4207 - mae: 1.0052 - mse: 1.4069 - val_loss: 0.8442 - val_mae: 0.8655 - val_mse: 0.8304
Epoch 4/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.2253 - mae: 0.9390 - mse: 1.2115 - val_loss: 0.7286 - val_mae: 0.8098 - val_mse: 0.7148
Epoch 5/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.0692 - mae: 0.8788 - mse: 1.0554 - val_loss: 0.6313 - val_mae: 0.7567 - val_mse: 0.6175
Epoch 6/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.9465 - mae: 0.8255 - mse: 0.9327 - val_loss: 0.5426 - val_mae: 0.7025 - val_mse: 0.5289
Epoch 7/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.8363 - mae: 0.7734 - mse: 0.8226 - val_loss: 0.4697 - val_mae: 0.6534 - val_mse: 0.4559
Epoch 8/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.7398 - mae: 0.7229 - mse: 0.7261 - va

In [27]:
# Predictions
y_pred = test_model_log.predict(X_test, verbose=0).flatten()

# Errors
error_log = y_pred - y_test
abs_error_log = np.abs(error_log)

# Statistics
results_log = {
    "Severity": "Lognormal",
    "Observations": len(y_test),
    "Training Time (s)": training_time,
    "Bias": np.mean(error_log),
    "MAE": np.mean(abs_error_log),
    "MSE": np.mean(error_log**2),
    "RMSE": np.sqrt(np.mean(error_log**2)),
    "95% AE": np.quantile(abs_error_log, 0.95),
    "99% AE": np.quantile(abs_error_log, 0.99),
    "Max AE": np.max(abs_error_log),
    "R2": r2_score(y_test, y_pred)
}

results_log_df = pd.DataFrame([results_log])

print(results_log_df)

    Severity  Observations  Training Time (s)      Bias       MAE       MSE  \
0  Lognormal        120000         639.366095  0.000047  0.003794  0.000026   

       RMSE    95% AE    99% AE    Max AE        R2  
0  0.005142  0.010619  0.016024  0.034223  0.999774  


### Test configuration 2

In [29]:
# Define different configurations
test_layers = [512, 256, 128, 64]

test_activation = 'relu'
test_l2 = 1e-4
test_dropout = 0.1
test_batch_norm = True
test_learning = 1e-5

In [59]:
import time

EPOCHS = 1000
BATCH_SIZE = 512

# Build the model
test_model_log = build_model(input_dim=X_train.shape[1],
                         nr_neurons=test_layers,
                         reg_param=test_l2,
                         activation=test_activation,
                         use_batch_norm=test_batch_norm,
                         dropout_rate=test_dropout,
                         learning_rate=test_learning)
# Measure training time
start_time = time.perf_counter()

# Train the model on the entire dataset
history_log = test_model_log.fit(X_train, y_train,
                         epochs=EPOCHS,
                         batch_size=BATCH_SIZE,
                         validation_split=0.2,
                         shuffle=True,          
                         callbacks=[early_stop],
                         verbose=1)

training_time = time.perf_counter() - start_time

print(f"Training time: {training_time:.2f} seconds")

Epoch 1/1000
742/750 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.4174 - mae: 1.2593 - mse: 2.3569

2026-07-02 14:50:17.753633: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-07-02 14:50:17.753834: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 2.4129 - mae: 1.2582 - mse: 2.3524 - val_loss: 0.9995 - val_mae: 0.9181 - val_mse: 0.9391
Epoch 2/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 1.5302 - mae: 1.0280 - mse: 1.4698 - val_loss: 0.8413 - val_mae: 0.8534 - val_mse: 0.7809
Epoch 3/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 1.2466 - mae: 0.9316 - mse: 1.1862 - val_loss: 0.7091 - val_mae: 0.7841 - val_mse: 0.6487
Epoch 4/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 1.0573 - mae: 0.8539 - mse: 0.9970 - val_loss: 0.6030 - val_mae: 0.7197 - val_mse: 0.5427
Epoch 5/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 0.9088 - mae: 0.7841 - mse: 0.8485 - val_loss: 0.5012 - val_mae: 0.6487 - val_mse: 0.4409
Epoch 6/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.7825 - mae: 0.7165 - mse: 0.7221 - val_loss: 0.4133 - val_mae: 0.5804 - val_mse: 0.3530
Epoch 7/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.6767 - mae: 0.6530 - mse: 0.6164 - va

In [60]:
# Predictions
y_pred = test_model_log.predict(X_test, verbose=0).flatten()

# Errors
error_log = y_pred - y_test
abs_error_log = np.abs(error_log)

# Statistics
results_log = {
    "Severity": "Lognormal",
    "Observations": len(y_test),
    "Training Time (s)": training_time,
    "Bias": np.mean(error_log),
    "MAE": np.mean(abs_error_log),
    "MSE": np.mean(error_log**2),
    "RMSE": np.sqrt(np.mean(error_log**2)),
    "95% AE": np.quantile(abs_error_log, 0.95),
    "99% AE": np.quantile(abs_error_log, 0.99),
    "Max AE": np.max(abs_error_log),
    "R2": r2_score(y_test, y_pred)
}

results_log_df = pd.DataFrame([results_log])

print(results_log_df)

    Severity  Observations  Training Time (s)      Bias      MAE       MSE  \
0  Lognormal        120000         2610.53311  0.000613  0.00309  0.000019   

       RMSE   95% AE    99% AE    Max AE        R2  
0  0.004406  0.00956  0.014882  0.035494  0.999834  
